In [1]:
import pandas as pd

# Load the datasets
train_df = pd.read_csv('/content/train.csv')
validation_df = pd.read_csv('/content/validation.csv')
test_df = pd.read_csv('/content/test.csv')

print('Train DataFrame Head:')
display(train_df.head())

print('\nValidation DataFrame Head:')
display(validation_df.head())

print('\nTest DataFrame Head:')
display(test_df.head())

Train DataFrame Head:


,label,texts
0,1,"Ah, so I have been told ;)"
1,0,just honest
2,0,i'm trying to get into med school
3,1,The rhythm of your heart is music to my ears.
4,0,"hi, not bad, how about yours?"



Validation DataFrame Head:


,label,texts
0,0,sure
1,0,congratulations 🐤
2,1,I'd drop everything for you in a heartbeat.
3,0,i'm a little too old to be adopted
4,0,"alright, how are you?"



Test DataFrame Head:


,label,texts
0,1,oh yeah...
1,1,Aaargh... Mhmmmm
2,0,Well we matched on Harry lol what’s ur fav song
3,1,Hey ;)
4,0,Exactly! I think if I were to do college again...


In [2]:
print('Train DataFrame Info:')
train_df.info()
print('\nValidation DataFrame Info:')
validation_df.info()
print('\nTest DataFrame Info:')
test_df.info()

Train DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1584 entries, 0 to 1583
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   1584 non-null   int64 
 1   texts   1584 non-null   object
dtypes: int64(1), object(1)
memory usage: 24.9+ KB

Validation DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 212 entries, 0 to 211
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   212 non-null    int64 
 1   texts   212 non-null    object
dtypes: int64(1), object(1)
memory usage: 3.4+ KB

Test DataFrame Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 318 entries, 0 to 317
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   label   318 non-null    int64 
 1   texts   318 non-null    object
dtypes: int64(1), object(1)
memory usage: 5.1+ KB


In [3]:
print('Missing values in train_df["texts"]: ', train_df['texts'].isnull().sum())
print('Missing values in validation_df["texts"]: ', validation_df['texts'].isnull().sum())
print('Missing values in test_df["texts"]: ', test_df['texts'].isnull().sum())

# Fill any potential NaN values with an empty string
train_df['texts'] = train_df['texts'].fillna('')
validation_df['texts'] = validation_df['texts'].fillna('')
test_df['texts'] = test_df['texts'].fillna('')

# Convert text to lowercase
train_df['texts'] = train_df['texts'].apply(lambda x: x.lower())
validation_df['texts'] = validation_df['texts'].apply(lambda x: x.lower())
test_df['texts'] = test_df['texts'].apply(lambda x: x.lower())

print('\nAfter cleaning, first 5 texts from train_df:')
display(train_df['texts'].head())

Missing values in train_df["texts"]:  0
Missing values in validation_df["texts"]:  0
Missing values in test_df["texts"]:  0

After cleaning, first 5 texts from train_df:


,texts
0,"ah, so i have been told ;)"
1,just honest
2,i'm trying to get into med school
3,the rhythm of your heart is music to my ears.
4,"hi, not bad, how about yours?"


In [6]:
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab') # Added to resolve LookupError
import string
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Function to remove punctuation
def remove_punctuation(text):
    return text.translate(str.maketrans('', '', string.punctuation))

# Apply punctuation removal
train_df['texts'] = train_df['texts'].apply(remove_punctuation)
validation_df['texts'] = validation_df['texts'].apply(remove_punctuation)
test_df['texts'] = test_df['texts'].apply(remove_punctuation)

# Tokenization and Stopword Removal
stop_words = set(stopwords.words('english'))

def tokenize_and_remove_stopwords(text):
    word_tokens = word_tokenize(text)
    filtered_sentence = [w for w in word_tokens if not w in stop_words]
    return ' '.join(filtered_sentence)

# Apply tokenization and stopword removal
train_df['texts'] = train_df['texts'].apply(tokenize_and_remove_stopwords)
validation_df['texts'] = validation_df['texts'].apply(tokenize_and_remove_stopwords)
test_df['texts'] = test_df['texts'].apply(tokenize_and_remove_stopwords)

print('\nAfter full preprocessing, first 5 texts from train_df:')
display(train_df['texts'].head())


After full preprocessing, first 5 texts from train_df:


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


,texts
0,ah told
1,honest
2,im trying get med school
3,rhythm heart music ears
4,hi bad


In [7]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Initialize TF-IDF Vectorizer
# We can adjust parameters like max_features, min_df, max_df, ngram_range for better performance
tfidf_vectorizer = TfidfVectorizer(max_features=5000, min_df=5, max_df=0.8)

# Fit the vectorizer on the training data and transform it
X_train_tfidf = tfidf_vectorizer.fit_transform(train_df['texts'])

# Transform validation and test data using the fitted vectorizer
X_val_tfidf = tfidf_vectorizer.transform(validation_df['texts'])
X_test_tfidf = tfidf_vectorizer.transform(test_df['texts'])

y_train = train_df['label']
y_val = validation_df['label']
y_test = test_df['label']

print(f'Shape of X_train_tfidf: {X_train_tfidf.shape}')
print(f'Shape of X_val_tfidf: {X_val_tfidf.shape}')
print(f'Shape of X_test_tfidf: {X_test_tfidf.shape}')

Shape of X_train_tfidf: (1584, 317)
Shape of X_val_tfidf: (212, 317)
Shape of X_test_tfidf: (318, 317)


In [11]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the SVM model
# Using 'linear' kernel as it often performs well on text data
svm_model = SVC(kernel='linear', random_state=42)
svm_model.fit(X_train_tfidf, y_train)

# Make predictions on the validation set
y_val_pred_svm = svm_model.predict(X_val_tfidf)

# Evaluate the SVM model on the validation set
print('SVM Validation Set Metrics:')
print(f'Accuracy: {accuracy_score(y_val, y_val_pred_svm):.4f}')
print(classification_report(y_val, y_val_pred_svm))

# Make predictions on the test set
y_test_pred_svm = svm_model.predict(X_test_tfidf)

# Evaluate the SVM model on the test set
print('\nSVM Test Set Metrics:')
print(f'Accuracy: {accuracy_score(y_test, y_test_pred_svm):.4f}')
print(classification_report(y_test, y_test_pred_svm))

SVM Validation Set Metrics:
Accuracy: 0.6934
              precision    recall  f1-score   support

           0       0.66      0.80      0.72       106
           1       0.75      0.58      0.66       106

    accuracy                           0.69       212
   macro avg       0.70      0.69      0.69       212
weighted avg       0.70      0.69      0.69       212


SVM Test Set Metrics:
Accuracy: 0.7264
              precision    recall  f1-score   support

           0       0.69      0.84      0.75       159
           1       0.79      0.62      0.69       159

    accuracy                           0.73       318
   macro avg       0.74      0.73      0.72       318
weighted avg       0.74      0.73      0.72       318



In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

# Initialize and train the Logistic Regression model
logistic_model = LogisticRegression(max_iter=1000, random_state=42)
logistic_model.fit(X_train_tfidf, y_train)

# Make predictions on the validation set
y_val_pred = logistic_model.predict(X_val_tfidf)

# Evaluate the model on the validation set
print('Validation Set Metrics:')
print(f'Accuracy: {accuracy_score(y_val, y_val_pred):.4f}')
print(classification_report(y_val, y_val_pred))

# Make predictions on the test set
y_test_pred = logistic_model.predict(X_test_tfidf)

# Evaluate the model on the test set
print('\nTest Set Metrics:')
print(f'Accuracy: {accuracy_score(y_test, y_test_pred):.4f}')
print(classification_report(y_test, y_test_pred))

Validation Set Metrics:
Accuracy: 0.6698
              precision    recall  f1-score   support

           0       0.64      0.76      0.70       106
           1       0.71      0.58      0.64       106

    accuracy                           0.67       212
   macro avg       0.68      0.67      0.67       212
weighted avg       0.68      0.67      0.67       212


Test Set Metrics:
Accuracy: 0.7327
              precision    recall  f1-score   support

           0       0.69      0.84      0.76       159
           1       0.79      0.63      0.70       159

    accuracy                           0.73       318
   macro avg       0.74      0.73      0.73       318
weighted avg       0.74      0.73      0.73       318



In [13]:
import joblib

# Define file paths for saving
model_path = 'logistic_regression_model.pkl'
vectorizer_path = 'tfidf_vectorizer.pkl'

# Save the Logistic Regression model
joblib.dump(logistic_model, model_path)
print(f'Logistic Regression model saved to {model_path}')

# Save the TF-IDF Vectorizer
joblib.dump(tfidf_vectorizer, vectorizer_path)
print(f'TF-IDF Vectorizer saved to {vectorizer_path}')

Logistic Regression model saved to logistic_regression_model.pkl
TF-IDF Vectorizer saved to tfidf_vectorizer.pkl


In [16]:
import zipfile
import os

# Files to be included in the zip archive
files_to_zip = ['logistic_regression_model.pkl', 'tfidf_vectorizer.pkl', 'app.py', 'requirements.txt']
zip_filename = 'flirt_detection_streamlit.zip'

# Create a ZipFile object
with zipfile.ZipFile(zip_filename, 'w') as zipf:
    for file in files_to_zip:
        if os.path.exists(file):
            zipf.write(file)
            print(f'Added {file} to {zip_filename}')
        else:
            print(f'Warning: {file} not found and could not be added to the zip.')

print(f'\nSuccessfully created {zip_filename}. You can now download this file.')

Added logistic_regression_model.pkl to flirt_detection_streamlit.zip
Added tfidf_vectorizer.pkl to flirt_detection_streamlit.zip
Added app.py to flirt_detection_streamlit.zip
Added requirements.txt to flirt_detection_streamlit.zip

Successfully created flirt_detection_streamlit.zip. You can now download this file.
